以下是基于 NicheCompass 原始代码运行指定乳腺癌数据集的完整代码

In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Run original NicheCompass model on ViHBC integrated dataset.
Uses existing cache at /home/zhangjunyi/xiangmu/nichecompass-main/cache_nichenet.
"""

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import scanpy as sc

# 可选：如果没有空间图时自动构图
try:
    import squidpy as sq
    HAS_SQUIDPY = True
except Exception:
    HAS_SQUIDPY = False

# -----------------------------
# 0) 路径配置
# -----------------------------
PROJECT_ROOT = Path("/home/zhangjunyi/xiangmu/nichecompass-main")
DATA_PATH = PROJECT_ROOT / "datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
OUT_DIR = PROJECT_ROOT / "outputs/vihbc_original_nichecompass"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 【修改点】直接使用你已有的缓存目录
CACHE_DIR = PROJECT_ROOT / "cache_nichenet"

SRC_DIR = PROJECT_ROOT / "src"
PREFERRED_PKG_DIR = SRC_DIR / "nichecompass_copy"
SPACE_PKG_DIR = SRC_DIR / "nichecompass copy"

# -----------------------------
# 1) 导入原始 NicheCompass 包 (包括 utils)
# -----------------------------
def import_original_nichecompass():
    final_pkg_name = "nichecompass"
    tmp_root = PROJECT_ROOT / "_tmp_import"
    
    # 确定源目录
    if PREFERRED_PKG_DIR.exists():
        src_pkg = PREFERRED_PKG_DIR
    elif SPACE_PKG_DIR.exists():
        src_pkg = SPACE_PKG_DIR
    else:
        raise FileNotFoundError(f"未找到原始模型目录：{PREFERRED_PKG_DIR} 或 {SPACE_PKG_DIR}")

    # 复制到临时目录并重命名为标准的 "nichecompass"
    import shutil
    tmp_root.mkdir(parents=True, exist_ok=True)
    final_pkg = tmp_root / final_pkg_name
    
    if final_pkg.exists():
        shutil.rmtree(final_pkg)
    shutil.copytree(src_pkg, final_pkg)

    if str(tmp_root) not in sys.path:
        sys.path.insert(0, str(tmp_root))

    # 导入模型和工具函数
    from nichecompass.models import NicheCompass
    import nichecompass.utils as nc_utils
    
    return NicheCompass, nc_utils


# -----------------------------
# 2) 一些默认 key
# -----------------------------
COUNTS_KEY = "counts"
ADJ_KEY = "spatial_connectivities"

GP_NAMES_KEY = "nichecompass_gp_names"
ACTIVE_GP_NAMES_KEY = "nichecompass_active_gp_names"
GP_TARGETS_MASK_KEY = "nichecompass_gp_targets"
GP_TARGETS_CATEGORIES_MASK_KEY = "nichecompass_gp_targets_categories"
GP_SOURCES_MASK_KEY = "nichecompass_gp_sources"
GP_SOURCES_CATEGORIES_MASK_KEY = "nichecompass_gp_sources_categories"

LATENT_KEY = "nichecompass_latent"
LEIDEN_KEY = "nichecompass_latent_leiden"

# -----------------------------
# 3) 工具函数
# -----------------------------
def ensure_counts(adata, counts_key=COUNTS_KEY):
    if counts_key not in adata.layers:
        warnings.warn(f"adata.layers['{counts_key}'] 不存在，自动使用 adata.X 作为输入。")
        return None
    return counts_key

def ensure_spatial_graph(adata, adj_key=ADJ_KEY, n_neighbors=8):
    if adj_key in adata.obsp:
        return
    if not HAS_SQUIDPY:
        raise RuntimeError(f"缺少 {adj_key} 且未安装 squidpy。")
    if "spatial" not in adata.obsm:
        raise RuntimeError(f"缺少 spatial 坐标。")

    sq.gr.spatial_neighbors(
        adata,
        coord_type="generic",
        spatial_key="spatial",
        n_neighs=n_neighbors
    )
    adata.obsp[adj_key] = adata.obsp[adj_key].maximum(adata.obsp[adj_key].T)

def add_gp_priors_if_missing(adata, nc_utils, cache_dir):
    """
    从现有缓存加载 GP 先验。
    """
    # 检查是否已经有了
    has_gp = (
        GP_NAMES_KEY in adata.uns and
        GP_TARGETS_MASK_KEY in adata.varm and
        GP_SOURCES_MASK_KEY in adata.varm
    )
    
    if has_gp:
        print("检测到 adata 中已包含 GP 先验，跳过加载。")
        return adata

    print(f"从现有缓存加载 GP 先验: {cache_dir}")
    
    lr_path = cache_dir / "nichenet_lr_network.csv"
    lt_path = cache_dir / "nichenet_ligand_target_matrix.csv"

    if not (lr_path.exists() and lt_path.exists()):
        raise FileNotFoundError(f"在 {cache_dir} 中未找到缓存文件，请检查路径。")

    gp_dict = nc_utils.extract_gp_dict_from_nichenet_lrt_interactions(
        species="human",
        version="v2",
        keep_target_genes_ratio=0.25,
        max_n_target_genes_per_gp=50,
        load_from_disk=True,
        lr_network_file_path=str(lr_path),
        ligand_target_matrix_file_path=str(lt_path),
        plot_gp_gene_count_distributions=False,
    )

    print("将 GP 先验写入 adata...")
    nc_utils.add_gps_from_gp_dict_to_adata(
        gp_dict=gp_dict,
        adata=adata,
    )
    
    print("GP 先验添加完成。")
    return adata

def print_basic_info(adata):
    print("===== AnnData 概览 =====")
    print(f"n_obs: {adata.n_obs}, n_vars: {adata.n_vars}")
    print(f"layers keys: {list(adata.layers.keys())[:10]}")
    print(f"obsp keys: {list(adata.obsp.keys())[:10]}")
    print(f"varm keys: {list(adata.varm.keys())[:10]}")
    print(f"uns keys: {list(adata.uns.keys())[:20]}")
    print("=======================")


# -----------------------------
# 4) 主流程
# -----------------------------
def main():
    np.random.seed(0)
    sc.settings.verbosity = 2
    sc.settings.set_figure_params(dpi=120, facecolor="white")

    # 4.1 导入原始模型 (和 utils)
    NicheCompass, nc_utils = import_original_nichecompass()

    # 4.2 读数据
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"数据文件不存在：{DATA_PATH}")
    adata = sc.read_h5ad(DATA_PATH)
    print_basic_info(adata)

    # 4.3 预处理
    counts_key = ensure_counts(adata, COUNTS_KEY)
    ensure_spatial_graph(adata, ADJ_KEY, n_neighbors=8)
    adata = add_gp_priors_if_missing(adata, nc_utils, CACHE_DIR)

    # 4.4 初始化模型
    print("初始化 NicheCompass 模型...")
    model = NicheCompass(
        adata=adata,
        counts_key=counts_key,
        adj_key=ADJ_KEY,
        gp_names_key=GP_NAMES_KEY,
        active_gp_names_key=ACTIVE_GP_NAMES_KEY,
        gp_targets_mask_key=GP_TARGETS_MASK_KEY,
        gp_targets_categories_mask_key=GP_TARGETS_CATEGORIES_MASK_KEY,
        gp_sources_mask_key=GP_SOURCES_MASK_KEY,
        gp_sources_categories_mask_key=GP_SOURCES_CATEGORIES_MASK_KEY,
        latent_key=LATENT_KEY,
        conv_layer_encoder="gatv2conv",
        active_gp_thresh_ratio=0.01,
        seed=0,
        use_cuda_if_available=True,
    )

    # 4.5 训练
    print("开始训练模型...")
    model.train(
        n_epochs=200,
        n_epochs_all_gps=25,
        lr=1e-3,
        lambda_edge_recon=5e5,
        lambda_gene_expr_recon=300.0,
        lambda_l1_masked=0.0,
        lambda_l1_addon=30.0,
        edge_val_ratio=0.1,
        node_val_ratio=0.1,
        edge_batch_size=256,
        n_sampled_neighbors=-1,
        use_cuda_if_available=True,
        verbose=False,
    )
    print("训练完成。")

    # 4.6 生态位识别
    print("进行下游分析...")
    sc.pp.neighbors(model.adata, use_rep=LATENT_KEY, key_added=LATENT_KEY)
    sc.tl.umap(model.adata, neighbors_key=LATENT_KEY)
    sc.tl.leiden(
        adata=model.adata,
        resolution=0.4,
        key_added=LEIDEN_KEY,
        neighbors_key=LATENT_KEY,
    )

    # 4.7 保存结果
    # out_h5ad = OUT_DIR / "Human_breast_cancer_integrated_nichecompass_result.h5ad"
    # model.adata.write_h5ad(out_h5ad)
    # print(f"[Saved] 结果文件: {out_h5ad}")

    # model_dir = OUT_DIR / "model_ckpt"
    # model.save(dir_path=str(model_dir), overwrite=True, save_adata=False)
    # print(f"[Saved] 模型文件: {model_dir}")

    # 4.8 画图
    print("生成可视化图片...")
    fig1 = sc.pl.umap(model.adata, color=[LEIDEN_KEY], return_fig=True, show=False)
    fig1.savefig(OUT_DIR / "umap_niche_leiden.png", bbox_inches="tight")

    if "spatial" in model.adata.obsm:
        fig2 = sc.pl.embedding(
            model.adata, basis="spatial", color=[LEIDEN_KEY], 
            return_fig=True, show=False, spot_size=200
        )
        fig2.savefig(OUT_DIR / "spatial_niche_leiden.png", bbox_inches="tight")

    print("===== 运行完成 =====")
    print("生态位统计:")
    print(model.adata.obs[LEIDEN_KEY].value_counts())


if __name__ == "__main__":
    main()

===== AnnData 概览 =====
n_obs: 3798, n_vars: 36601
layers keys: []
obsp keys: []
varm keys: []
uns keys: ['spatial']
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
从现有缓存加载 GP 先验: /home/zhangjunyi/xiangmu/nichecompass-main/cache_nichenet
将 GP 先验写入 adata...
GP 先验添加完成。
初始化 NicheCompass 模型...
--- INITIALIZING NEW NETWORK MODULE: VARIATIONAL GENE PROGRAM GRAPH AUTOENCODER ---
LOSS -> include_edge_recon_loss: True, include_gene_expr_recon_loss: True, rna_recon_loss: nb
NODE LABEL METHOD -> one-hop-norm
ACTIVE GP THRESHOLD RATIO -> 0.01
LOG VARIATIONAL -> True
ONE HOP GCN NORM RNA NODE LABEL AGGREGATOR
ENCODER -> n_input: 36601, n_cat_covariates_embed_input: 0, n_hidden: 1326, n_latent: 1226, n_addon_latent: 100, n_fc_layers: 1, n_layers: 1, conv_layer: gatv2conv, n_attention_heads: 4, dropout_rate: 0.0, 
COSINE SIM 